In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import pickle
import sys
sys.path.append("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/")
from benchmarker import Benchmarker, recompute_aggregate_scores

/mnt/datadisk/lizhongzhan/miniconda3/envs/benchmark_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/datadisk/lizhongzhan/miniconda3/envs/benchmark_env/lib/python3.10/site-packages/umap/__init__.py:9: ImportWarning: Tensorflow not installed; ParametricUMAP will be unavailable
  warn(


In [2]:
import os
os.chdir("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Mouse_brain_unpaired/")

Prepare data

In [3]:
rna = sc.read_h5ad("rna.h5ad")
atac = sc.read_h5ad("atac.h5ad")
gs = sc.read_h5ad("gs.h5ad")
gs.var_names = [i.lower().capitalize() for i in gs.var_names ]

In [4]:
# comm_gene = pd.Index(set(rna.var_names) & set(gs.var_names))
# rna = rna[:,comm_gene]
# gs = gs[:,comm_gene]

In [5]:
(gs.obs_names == atac.obs_names).all()

np.True_

In [6]:
import anndata as ad
import h5py
import numpy as np
from scipy import sparse

def h5ad_to_h5(adata, output_file: str):

    if adata.raw is not None and adata.raw.X is not None:
        X = adata.raw.X
        features = np.asarray(adata.raw.var_names, dtype=str)
    else:
        X = adata.X
        features = np.asarray(adata.var_names, dtype=str)

    barcodes = np.asarray(adata.obs_names, dtype=str)

    spatial = None
    if "spatial" in adata.obsm:
        spatial = np.asarray(adata.obsm["spatial"], dtype=np.float32)

    if sparse.issparse(X):
        X = X.tocsr()
    else:
        X = sparse.csr_matrix(X)

    if X.data.size == 0 or (np.all(X.data >= 0) and np.all(np.isclose(X.data, np.round(X.data)))):
        X.data = X.data.astype(np.int32, copy=False)
    else:
        X.data = X.data.astype(np.float32, copy=False)

    with h5py.File(output_file, "w") as f:
        g = f.create_group("matrix")

        g.create_dataset("data", data=X.data,
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("indices", data=X.indices.astype(np.int32, copy=False),
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("indptr", data=X.indptr.astype(np.int64, copy=False),
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("shape", data=np.asarray(X.shape, dtype=np.int64))

        g.create_dataset("barcodes", data=np.array(barcodes, dtype="S"))
        g.create_dataset("features", data=np.array(features, dtype="S"))

        if spatial is not None:
            g.create_dataset(
                "spatial",
                data=spatial,
                compression="gzip",
                compression_opts=4,
                shuffle=True
            )

In [7]:
# h5ad_to_h5(rna, output_file="rna.h5")
# h5ad_to_h5(atac, output_file="atac.h5")
# h5ad_to_h5(gs, output_file="gs.h5")

In [8]:
bm = Benchmarker(R_conda_env="Rbase")

Run evaluation methods

In [9]:
# data_folder = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Mouse_brain_unpaired/"
# bm.run(methods=["Seurat_RPCA", "Seurat_CCA", "BindSC", "Monae"],
#        RNA_file_path=data_folder+"rna.h5",
#        ATAC_file_path=data_folder+"/atac.h5",
#        ADT_file_path=data_folder+"/gs.h5", # path for gene score
#        n_cluster=6,
#        conda_envs={"Monae": "scSLAT", "SIMBA": "scSLAT", "SCALEX": "scSLAT", "GLUE": "scSLAT",
#        "MaxFuse": "scSLAT", "LIGER": "scSLAT", "scConfluence": "cell2loc_env",},
#        save_path="/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsUnpaired/Mouse_brain")

In [10]:
# res = pd.read_csv("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsUnpaired/Mouse_brain/seurat_cca.csv", index_col=0)
# combined = sc.concat([rna, atac], label="omics")
# combined.obsm["spatial"][:,0] = combined.obsm["spatial"][:,0]*-1
# combined.obsm["spatial"][:,1] = combined.obsm["spatial"][:,1]*-1

In [11]:
# res.columns = ["UMAP1", "UMAP2", "cluster"]
# combined.obsm["X_umap"] = np.array(res[["UMAP1","UMAP2"]])
# combined.obs["cluster"] = [str(i) for i in list(res["cluster"])]

In [12]:
sc.set_figure_params(dpi=100, figsize=(4,4), facecolor="white")

In [13]:
# sc.pl.umap(combined, color=["cluster", "omics"])
# t_rna = combined[combined.obs["omics"]=="0",]
# t_atac = combined[combined.obs["omics"]=="1",]
# sc.pl.embedding(t_rna, color="cluster", size=100, basis="spatial")
# sc.pl.embedding(t_atac, color="cluster", size=100, basis="spatial")

Evaluation

In [14]:
result_folder = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsUnpaired/Mouse_brain/"
methods = ["GLUE", "Monae", "SIMBA", "SCALEX", "MaxFuse", "LIGER", "scConfluence", "Seurat_CCA",
"Seurat_RPCA", "BindSC", "switch"]
res = bm.read_result(path=result_folder,
                     methods=methods,
                     reindex=False)

In [15]:
combined = sc.concat([rna, gs], label="omics")

In [16]:
rna_annot = pd.read_csv("rna_annot.csv",index_col=1)
rna_annot = rna_annot.reindex(rna.obs_names)
atac_annot = pd.read_csv("atac_annot.csv",index_col=1)
atac_annot = atac_annot.reindex(atac.obs_names)
annot = pd.concat([rna_annot, atac_annot], axis=0)
annot = annot.reindex(combined.obs_names)
combined.obs["cell_type"] = annot["structure"].astype(str).copy()

In [17]:
# metrics = bm.cal_metrics(adata=combined, batch_key="omics", label_key="cell_type",
#                          res_dict=res, methods=methods, verbose=True, rep=1,
#                          min_max_scale=False,
#                          save=f"{result_folder}/metrics.pkl")

In [18]:
with open(f"{result_folder}/metrics.pkl", "rb") as f:
    metrics = pickle.load(f)
metric = metrics[0]

In [19]:
bm.set_plot_params(params_dict={"figure.dpi": 300},
# font_file_path="/mnt/datadisk/lizhongzhan/SpaMultiOmics/Helvetica.ttf"
)

In [20]:
figure_save_dir = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/figures/Multi_omics_unpaired/Mouse_brain"

In [21]:
metric = metrics[0]
metric.index = metric.index.map({
    i: i if i != "switch" else "SWITCH" for i in metric.index
})

In [22]:
metric = recompute_aggregate_scores(metric)
metric["Total"][:-1] = metric["Batch correction"][:-1] * 0.4 + metric["Bio conservation"][:-1] * 0.6

In [23]:
# bm.plot_heatmap(metric_df=metric, total_name="Total",
#                 save=f"{figure_save_dir}/summary_heatmap.pdf",
#                 # show_top=7,
#                 # show_bottom=0,
#                 # insert_marker_row = 8,
#                 )

In [24]:
from benchmarker import split_adata, transform_coord
import numpy as np
spatial = [i.obsm["spatial"] for i in split_adata(combined, batch_key="omics")]
spatial = transform_coord(spatial, vertical=True, axis="y", horizontal=False, angle=0)

In [25]:
spatial_methods = ["switch"]
bg_dict = {i:"#D4B483" if i in spatial_methods else "#5873a4" for i in bm.all_methods }
bg_dict["RNA"] = "#97a4af"
bg_dict["ATAC"] = "#97a4af"
bg_dict["Annotation"] = "#97a4af"
bg_dict["SWITCH"] = "#D4B483"
bg_dict["Modality"] = "#97a4af"
bg_dict["Cell type"] = "#97a4af"

In [26]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict=res["Cluster"],
#                 figsize=(14, 4.2),
#                 frameon=True,
#                 inner_gs_row=2,
#                 inner_gs_col=1,
#                 size=20,
#                 ncol=7,
#                 xlabel=["SWITCH", "GLUE", "Monae", "SCALEX", "scConfluence", "MaxFuse", "BindSC"],
#                 ylabel=None, #["E15", "E18"],
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 order=["switch", "GLUE", "Monae", "SCALEX", "scConfluence", "MaxFuse", "BindSC"],
#                 outer_row_hspace=0,
#                 outer_col_wspace=0.02,
#                 inner_common_camp=True,
#                 palette=palette,
#                 # inner_col_wspace = -0.12,
#                 ylabel_pad = 0.02,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.013, save_dpi=600,
#                 save=f"{figure_save_dir}/spatial_methods.pdf",
#                 rasterized=True,
#                 )

In [32]:
from benchmarker import get_scatter_cmap
palette = get_scatter_cmap(sorted([str(i) for i in list(range(6))], reverse=False))

In [42]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict=res["Cluster"],
#                 figsize=(14, 8.6),
#                 frameon=True,
#                 inner_gs_row=2,
#                 inner_gs_col=1,
#                 size=20,
#                 ncol=7,
#                 xlabel=["SWITCH", "GLUE", "Monae", "SCALEX", "scConfluence", "MaxFuse", "BindSC", "Seurat_RPCA", "SIMBA", "Seurat_CCA",
#                 "LIGER"],
#                 ylabel=["RNA","ATAC"],
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 order=["switch", "GLUE", "Monae", "SCALEX", "scConfluence", "MaxFuse", "BindSC", "Seurat_RPCA", "SIMBA", "Seurat_CCA",
#                 "LIGER"],
#                 outer_row_hspace=0.12,
#                 outer_col_wspace=0.02,
#                 inner_common_camp=True,
#                 palette=palette,
#                 # inner_col_wspace = -0.12,
#                 ylabel_pad = 0.0168,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.0125, save_dpi=600,
#                 save=f"{figure_save_dir}/spatial_methods_all.pdf",
#                 rasterized=True,
#                 )

In [1]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict={"annot": np.array(combined.obs["cell_type"]).reshape(-1,1)},
#                 figsize=(1.97, 4.2),
#                 frameon=True,
#                 inner_gs_row=2, inner_gs_col=1,
#                 size=20,
#                 ncol=1,
#                 xlabel=["Annotation"],
#                 ylabel=["RNA", "ATAC"],
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 outer_row_hspace=0.15,
#                 outer_col_wspace=0.1,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.013,
#                 ylabel_pad=0.0165,
#                 save_dpi=600,
#                 palette=palette,
#                 # save=f"{figure_save_dir}/spatial_annot.pdf"
#                 )

In [90]:
palette_annot = {"vl": "#1f77b4",
"cp": "#ff7f0e",
"ccg": "#2ca02c",
"ls": "#d62728",
"ctx": "#9467bd",
"aca": "#8c564b"}

In [91]:
combined.obs["cell_type_annoted"] = combined.obs["cell_type"].map({
    "1": "vl",
    "2": "cp",
    "3": "ccg",
    "4": "ls",
    "5": "ctx",
    "6": "aca"
})

In [92]:
res["Batch"] = {}
for m in methods:
    res["Batch"][m] = np.array(combined.obs["omics"]).astype(str)

In [93]:
# bm.plot_umap(embed_dict=res["UMAP"],
#              batch_dict=res["Batch"],
#              annot_list=list(combined.obs["cell_type_annoted"]),
#              figsize=(14, 4.2),
#              frameon=True,
#              inner_gs_row=2,
#              inner_gs_col=1,
#              size=7.5,
#              ncol=7,
#              xlabel=["SWITCH", "GLUE", "Monae", "SCALEX", "scConfluence", "MaxFuse", "BindSC"],
#              only_show_top=False,
#              ylabel=["Modality", "Cell type"],
#              only_show_left=True,
#              background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#              order=["switch", "GLUE", "Monae", "SCALEX", "scConfluence", "MaxFuse", "BindSC"],
#              axis_width=1.2,
#              axis_color="lightgrey",
#              outer_col_wspace=0.05,
#              save_dpi=600,
#              ylabel_pad=0.0168,
#              xlabel_pad=0.013,
#              outer_row_hspace=0.22,
#              merge=True,
#              merge_margin_size=0.4,
#              palettes=[None, palette_annot],
#              save=f"{figure_save_dir}/umap_methods.pdf"
# )

In [94]:
# bm.plot_umap(embed_dict=res["UMAP"],
#              batch_dict=res["Batch"],
#              annot_list=list(combined.obs["cell_type_annoted"]),
#              figsize=(14, 8.2),
#              frameon=True,
#              inner_gs_row=2,
#              inner_gs_col=1,
#              size=7.5,
#              ncol=7,
#              xlabel=["SWITCH", "GLUE", "Monae", "SCALEX", "scConfluence", "MaxFuse", "BindSC", "Seurat_RPCA", "SIMBA", "Seurat_CCA",
#                 "LIGER"],
#              only_show_top=False,
#              ylabel=["Modality", "Cell type"],
#              only_show_left=True,
#              background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#              order=["switch", "GLUE", "Monae", "SCALEX", "scConfluence", "MaxFuse", "BindSC", "Seurat_RPCA", "SIMBA", "Seurat_CCA",
#                 "LIGER"],
#              axis_width=1.2,
#              axis_color="lightgrey",
#              outer_col_wspace=0.05,
#              save_dpi=600,
#              ylabel_pad=0.0168,
#              xlabel_pad=0.013,
#              outer_row_hspace=0.12,
#              merge=True,
#              merge_margin_size=0.4,
#              palettes=[None, palette_annot],
#              save=f"{figure_save_dir}/umap_methods_all.pdf"
# )

In [95]:
# bm.plot_legend(palette_annot, marker="o", ncol=1,
# save=f"{figure_save_dir}/annot_legend.pdf",)

In [96]:
# bm.plot_legend(["RNA", "ATAC"], marker="o", ncol=1,
# save=f"{figure_save_dir}/omics_legend.pdf",)

In [97]:
# bm.plot_legend(category_lst=[str(i) for i in range(6)],
#                 marker="o",
#                 ncol=6,
#                 save=f"{figure_save_dir}/cluster_legend.pdf",
#                 )